In [1]:
!pip install ultralytics opencv-python matplotlib seaborn scikit-learn
!pip install tqdm

In [21]:
import os
import shutil

# -------- PATHS --------
gt_file = "/mnt/DATA/IIT_DO_v7/gt/gt.txt"
image_dir = "/mnt/DATA/IIT_DO_v7/img1"

output_dir = "/mnt/DATA/IIT_DO_v7/new_dataset"
output_images = os.path.join(output_dir, "images")
output_labels = os.path.join(output_dir, "labels")

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)

# -------- IMAGE SIZE --------
IMG_WIDTH = 1920   # ⚠️ change if different
IMG_HEIGHT = 1080  # ⚠️ change if different

# -------- READ GT FILE --------
data = {}

with open(gt_file, "r") as f:
    for line in f:
        parts = line.strip().split(",")

        frame = int(parts[0])
        x, y, w, h = map(float, parts[2:6])

        if frame not in data:
            data[frame] = []

        data[frame].append((x, y, w, h))

# -------- PROCESS --------
for frame_id, boxes in data.items():

    # 🔥 IMPORTANT CHANGE HERE (zero padding)
    img_name = f"{frame_id:08d}.jpg"

    src_img_path = os.path.join(image_dir, img_name)

    if not os.path.exists(src_img_path):
        print(f"❌ Missing: {img_name}")
        continue

    # copy image
    dst_img_path = os.path.join(output_images, img_name)
    shutil.copy(src_img_path, dst_img_path)

    # create label
    label_name = f"{frame_id:08d}.txt"
    label_path = os.path.join(output_labels, label_name)

    with open(label_path, "w") as lf:
        for (x, y, w, h) in boxes:

            # Convert to YOLO
            x_center = (x + w / 2) / IMG_WIDTH
            y_center = (y + h / 2) / IMG_HEIGHT
            w_norm = w / IMG_WIDTH
            h_norm = h / IMG_HEIGHT

            class_id = 0

            lf.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

print("✅ Dataset created successfully!")

✅ Dataset created successfully!


In [22]:
import os
import shutil

# -------- PATHS --------
gt_file = "/mnt/DATA/IIT_DO_v2/gt/gt.txt"
image_dir = "/mnt/DATA/IIT_DO_v2/img1"

output_dir = "/mnt/DATA/IIT_DO_v2/new_dataset"
output_images = os.path.join(output_dir, "images")
output_labels = os.path.join(output_dir, "labels")

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)

# -------- IMAGE SIZE --------
IMG_WIDTH =  3840  # ⚠️ change if different
IMG_HEIGHT = 2160  # ⚠️ change if different

# -------- READ GT FILE --------
data = {}

with open(gt_file, "r") as f:
    for line in f:
        parts = line.strip().split(",")

        frame = int(parts[0])
        x, y, w, h = map(float, parts[2:6])

        if frame not in data:
            data[frame] = []

        data[frame].append((x, y, w, h))

# -------- PROCESS --------
for frame_id, boxes in data.items():

    # 🔥 IMPORTANT CHANGE HERE (zero padding)
    img_name = f"{frame_id:08d}.jpg"

    src_img_path = os.path.join(image_dir, img_name)

    if not os.path.exists(src_img_path):
        print(f"❌ Missing: {img_name}")
        continue

    # copy image
    dst_img_path = os.path.join(output_images, img_name)
    shutil.copy(src_img_path, dst_img_path)

    # create label
    label_name = f"{frame_id:08d}.txt"
    label_path = os.path.join(output_labels, label_name)

    with open(label_path, "w") as lf:
        for (x, y, w, h) in boxes:

            # Convert to YOLO
            x_center = (x + w / 2) / IMG_WIDTH
            y_center = (y + h / 2) / IMG_HEIGHT
            w_norm = w / IMG_WIDTH
            h_norm = h / IMG_HEIGHT

            class_id = 0

            lf.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

print("✅ Dataset created successfully!")

✅ Dataset created successfully!


In [23]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil
import zipfile

In [25]:
import os
DATASET1_PATH = "/mnt/DATA/IIT_DO_v2/new_dataset/images"
DATASET2_PATH = "/mnt/DATA/IIT_DO_v7/new_dataset/images"
DATASET3_PATH = "/mnt/DATA/fused_dataset/images"
LABEL1_PATH = "/mnt/DATA/IIT_DO_v2/new_dataset/labels"
LABEL2_PATH = "/mnt/DATA/IIT_DO_v7/new_dataset/labels"
LABEL3_PATH = "/mnt/DATA/fused_dataset/labels"

In [26]:
import os
import shutil
from sklearn.model_selection import train_test_split

# =========================================================
# DATASET 1
# =========================================================
DATASET1_IMAGES = "/mnt/DATA/IIT_DO_v2/new_dataset/images"
DATASET1_LABELS = "/mnt/DATA/IIT_DO_v2/new_dataset/labels"

# =========================================================
# DATASET 2
# =========================================================
DATASET2_IMAGES = "/mnt/DATA/IIT_DO_v7/new_dataset/images"
DATASET2_LABELS = "/mnt/DATA/IIT_DO_v7/new_dataset/labels"

# =========================================================
# DATASET 3 (ALREADY SPLIT)
# =========================================================
DATASET3_PATH = "/mnt/DATA/fused_dataset"

# =========================================================
# FINAL OUTPUT DATASET
# =========================================================
FINAL_DATASET = "/mnt/DATA/final_training_dataset"

# Create folders
for split in ["train", "val"]:

    os.makedirs(
        os.path.join(FINAL_DATASET, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(FINAL_DATASET, split, "labels"),
        exist_ok=True
    )

# =========================================================
# FUNCTION TO COPY FILES
# =========================================================
def copy_files(
    file_list,
    image_folder,
    label_folder,
    split_name,
    prefix
):

    for img_name in file_list:

        # -------------------------------
        # NEW UNIQUE FILENAMES
        # -------------------------------
        new_img_name = prefix + "_" + img_name

        label_name = os.path.splitext(img_name)[0] + ".txt"

        new_label_name = prefix + "_" + label_name

        # -------------------------------
        # SOURCE PATHS
        # -------------------------------
        src_img = os.path.join(image_folder, img_name)

        src_label = os.path.join(label_folder, label_name)

        # -------------------------------
        # DESTINATION PATHS
        # -------------------------------
        dst_img = os.path.join(
            FINAL_DATASET,
            split_name,
            "images",
            new_img_name
        )

        dst_label = os.path.join(
            FINAL_DATASET,
            split_name,
            "labels",
            new_label_name
        )

        # -------------------------------
        # COPY
        # -------------------------------
        shutil.copy(src_img, dst_img)
        shutil.copy(src_label, dst_label)

# =========================================================
# DATASET 1 SPLIT (80 TRAIN / 20 VAL)
# =========================================================
print("\nProcessing Dataset 1...")

dataset1_images = [

    f for f in os.listdir(DATASET1_IMAGES)

    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

train1, val1 = train_test_split(
    dataset1_images,
    test_size=0.20,
    random_state=42
)

copy_files(
    train1,
    DATASET1_IMAGES,
    DATASET1_LABELS,
    "train",
    "d1"
)

copy_files(
    val1,
    DATASET1_IMAGES,
    DATASET1_LABELS,
    "val",
    "d1"
)

print(f"Dataset1 Train: {len(train1)}")
print(f"Dataset1 Val  : {len(val1)}")

# =========================================================
# DATASET 2 SPLIT (80 TRAIN / 20 VAL)
# =========================================================
print("\nProcessing Dataset 2...")

dataset2_images = [

    f for f in os.listdir(DATASET2_IMAGES)

    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

train2, val2 = train_test_split(
    dataset2_images,
    test_size=0.20,
    random_state=42
)

copy_files(
    train2,
    DATASET2_IMAGES,
    DATASET2_LABELS,
    "train",
    "d2"
)

copy_files(
    val2,
    DATASET2_IMAGES,
    DATASET2_LABELS,
    "val",
    "d2"
)

print(f"Dataset2 Train: {len(train2)}")
print(f"Dataset2 Val  : {len(val2)}")

# =========================================================
# COPY DATASET 3 TRAIN + VAL
# =========================================================
print("\nCopying Dataset 3...")

for split in ["train", "val"]:

    image_folder = os.path.join(
        DATASET3_PATH,
        split,
        "images"
    )

    label_folder = os.path.join(
        DATASET3_PATH,
        split,
        "labels"
    )

    image_files = [

        f for f in os.listdir(image_folder)

        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    copy_files(
        image_files,
        image_folder,
        label_folder,
        split,
        "d3"
    )

    print(f"Dataset3 {split}: {len(image_files)}")

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\n===================================")
print("FINAL DATASET CREATED")
print("===================================")

for split in ["train", "val"]:

    image_count = len(
        os.listdir(
            os.path.join(FINAL_DATASET, split, "images")
        )
    )

    label_count = len(
        os.listdir(
            os.path.join(FINAL_DATASET, split, "labels")
        )
    )

    print(f"\n{split.upper()}")

    print(f"Images : {image_count}")
    print(f"Labels : {label_count}")

print(f"\nSaved at:\n{FINAL_DATASET}")


Processing Dataset 1...
Dataset1 Train: 1896
Dataset1 Val  : 474

Processing Dataset 2...
Dataset2 Train: 11592
Dataset2 Val  : 2898

Copying Dataset 3...
Dataset3 train: 4604
Dataset3 val: 616

FINAL DATASET CREATED

TRAIN
Images : 18092
Labels : 18092

VAL
Images : 3988
Labels : 3988

Saved at:
/mnt/DATA/final_training_dataset


In [27]:
yaml_content = """
path: /mnt/DATA/final_training_dataset

train: train/images
val: val/images

nc: 1

names:
  0: drone
"""

# =========================================================
# SAVE YAML FILE
# =========================================================
yaml_path = "/mnt/DATA/final_training_dataset/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"data.yaml created at:\n{yaml_path}")

data.yaml created at:
/mnt/DATA/final_training_dataset/data.yaml


In [36]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
data="/mnt/DATA/final_training_dataset/data.yaml",
epochs=30,
imgsz=1024,
batch=16,
scale=0.5,
mosaic=1.0,
mixup=0.2
)

New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.34 🚀 Python-3.8.20 torch-2.4.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16006MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/DATA/final_training_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f3d27824be0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [37]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
results = model.val(save_json=True, plots=True)
model.predict(
source="/mnt/DATA/final_training_dataset/val/images",
save=True
)

Ultralytics 8.4.34 🚀 Python-3.8.20 torch-2.4.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16006MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6503.8±2669.6 MB/s, size: 182.4 KB)
val: Scanning /mnt/DATA/final_training_dataset/val/labels.cache... 3988 images, 88 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3988/3988 1.1Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 250/250 8.5it/s 29.6s<0.2s
                   all       3988       3903      0.988      0.923       0.95      0.878
Speed: 0.7ms preprocess, 5.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /mnt/DATA/runs/detect/val3
mAP50: 0.950104059176846
mAP50-95: 0.8775538292218794
Precision: 0.987655451873252
Recall: 0.9226236228542147
Ultralytics 8.4.34 🚀 Python-3.8.20 torch-2.4.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16006MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, r

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'drone'}
 obb: None
 orig_img: array([[[152, 150, 132],
         [157, 155, 137],
         [160, 158, 140],
         ...,
         [146, 147, 131],
         [147, 148, 132],
         [148, 149, 133]],
 
        [[157, 155, 137],
         [156, 155, 135],
         [158, 156, 138],
         ...,
         [145, 146, 130],
         [146, 147, 131],
         [147, 148, 132]],
 
        [[158, 157, 137],
         [154, 153, 132],
         [154, 153, 133],
         ...,
         [147, 148, 132],
         [147, 148, 132],
         [146, 147, 131]],
 
        ...,
 
        [[ 44,  65,  73],
         [ 54,  75,  83],
         [ 49,  71,  77],
         ...,
         [ 74,  80,  75],
         [ 74,  81,  74],
         [ 76,  83,  76]],
 
        [[ 48,  71,  79],
         [ 34,  57,  65],
         [ 35,  59,  65],
         ...,
         [ 79,  85,

In [1]:
from ultralytics import YOLO

# replace with your actual best weights path
model = YOLO("/mnt/DATA/runs/detect/train5/weights/best.pt")
results = model.predict(
    source="/mnt/DATA/split_rgb",  # or val/images
    save=True,     # saves images with bounding boxes
    conf=0.25      # confidence threshold
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1439 /mnt/DATA/split_rgb/00000007.jpg: 1024x928 (no detections), 34.7ms
image 2/1439 /mnt/DATA/split_rgb/00000008.jpg: 1024x928 (no detections), 8.8ms
image 3/1439 /mnt/DATA/split_rgb/00000009.jpg: 1024x928 (no detections), 8.8ms
image 4/1439 /mnt/DATA/split_rgb/00000010.jpg: 1024x928 (no detections), 9.0ms
image 5/1439 /mnt/DATA/split_rgb/00000011.jpg: 1024x928 (no detections), 9.4ms
image 6/1439 /mnt/DATA/split_rgb/00000016.jpg: 1024x928 (no de

In [2]:
from ultralytics import YOLO

# replace with your actual best weights path
model = YOLO("/mnt/DATA/runs/detect/train5/weights/best.pt")
results = model.predict(
    source="/mnt/DATA/frames2",  # or val/images
    save=True,     # saves images with bounding boxes
    conf=0.25      # confidence threshold
)


image 1/408 /mnt/DATA/frames2/frame_00000.jpg: 832x1024 (no detections), 35.3ms
image 2/408 /mnt/DATA/frames2/frame_00001.jpg: 832x1024 (no detections), 8.2ms
image 3/408 /mnt/DATA/frames2/frame_00002.jpg: 832x1024 (no detections), 8.0ms
image 4/408 /mnt/DATA/frames2/frame_00003.jpg: 832x1024 (no detections), 7.9ms
image 5/408 /mnt/DATA/frames2/frame_00004.jpg: 832x1024 (no detections), 7.9ms
image 6/408 /mnt/DATA/frames2/frame_00005.jpg: 832x1024 (no detections), 9.0ms
image 7/408 /mnt/DATA/frames2/frame_00006.jpg: 832x1024 (no detections), 9.0ms
image 8/408 /mnt/DATA/frames2/frame_00007.jpg: 832x1024 (no detections), 8.4ms
image 9/408 /mnt/DATA/frames2/frame_00008.jpg: 832x1024 (no detections), 7.9ms
image 10/408 /mnt/DATA/frames2/frame_00009.jpg: 832x1024 (no detections), 7.9ms
image 11/408 /mnt/DATA/frames2/frame_00010.jpg: 832x1024 (no detections), 7.9ms
image 12/408 /mnt/DATA/frames2/frame_00011.jpg: 832x1024 (no detections), 9.1ms
image 13/408 /mnt/DATA/frames2/frame_00012.jpg: